# Random Forest

Il Random Forest è un algoritmo di machine learning supervisionato basato su una collezionei alberi decisionali indipendenti, combinati per migliorare la accuratezza e la robustezza rispetto a singoli alberi.

In [31]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, StratifiedKFold
from pathlib import Path
import warnings
# Nascondo i warning
warnings.filterwarnings('ignore')

# Definisco il percorso dei file
FILE_PATH = Path('/Users/francesco/Tesi/BC-ML4/dataset/cleaned')

# Lista dei csv su cui fare training
datasets = {
    't2_medsam': FILE_PATH / 't2_medsam_masks.csv',
    't2_preprocessed': FILE_PATH / 't2_preprocessed_masks.csv',
    't2_original': FILE_PATH / 't2_original_masks.csv',
    'medsam_dynamic': FILE_PATH / 'medsam_dynamic.csv',
    'preprocessed_dynamic': FILE_PATH / 'preprocessed_dynamic.csv',
    'original_dynamic': FILE_PATH / 'original_dynamic.csv'
}

# Treining

In [32]:
def training(file_path, csv_name):
    # Leggo i csv
    df = pd.read_csv(file_path)

    # Filtro solo le pazienti con PR valido
    df_PRvalido = df[df['PR [SII]'].notna()].copy()

    # Vado a separare le features e target
    features = df_PRvalido.drop(columns=['Patient ID', 'lesion idx', 'tumor/benign',
                             'GRADE', 'ER [SII]', 'PR [SII]', 'HER2 [SII]',
                             'isTN', 'KI67 [%]', 'Breast'])

    # Prendo solo PR [SII], convertita in valori interi.
    target = df_PRvalido['PR [SII]'].astype(int)

    # Normalizzo i dati per evitare problemi di scala
    scaler = StandardScaler()
    features_scaled = scaler.fit_transform(features)

    # Istanzio il modello Random Forest
    rf = RandomForestClassifier(
        n_estimators=100,       # Numero di alberi
        max_depth=None,         # Profondità massima nessuna
        min_samples_split=2,    # Minimo campioni per split
        min_samples_leaf=1,     # Minimo campioni per foglia
        random_state=42
    )

    # Alleno il modello sui dati normalizzati
    rf.fit(features_scaled, target)

    # Creo la cross-validation a 5 fold stratificata
    cv = StratifiedKFold(n_splits=15, shuffle=True, random_state=42)
    scores = cross_val_score(rf, features_scaled, target, cv=cv, scoring='accuracy')

    # Ritorno risultati sintetici: media, deviazione standard e score per ogni fold
    return {
        'mean_accuracy': scores.mean(),     # Accuratezza media su tutte le fold
        'std_accuracy': scores.std(),       # Variabilità tra le fold
        'scores_per_fold': scores           # Accuratezza tra ciascun fold
    }

# Lettura dei file

In [33]:
results = {}
print("="*50 +"\nRandom Forest\n" + "="*50)
results = {}
for name, file_path in datasets.items():
    print(f"\n{name}")
    results[name] = training(file_path, name)
    print(f"Accuracy media: {results[name]['mean_accuracy']:.3f} ± {results[name]['std_accuracy']:.3f}")
    print(f"Scores per fold: {[f'{s:.3f}' for s in results[name]['scores_per_fold']]}")

Random Forest

t2_medsam
Accuracy media: 0.447 ± 0.154
Scores per fold: ['0.400', '0.600', '0.400', '0.400', '0.600', '0.200', '0.600', '0.500', '0.750', '0.250', '0.500', '0.500', '0.250', '0.500', '0.250']

t2_preprocessed
Accuracy media: 0.373 ± 0.172
Scores per fold: ['0.200', '0.400', '0.200', '0.600', '0.400', '0.200', '0.600', '0.500', '0.500', '0.000', '0.500', '0.500', '0.250', '0.500', '0.250']

t2_original
Accuracy media: 0.437 ± 0.154
Scores per fold: ['0.600', '0.400', '0.400', '0.200', '0.400', '0.200', '0.600', '0.500', '0.500', '0.250', '0.500', '0.750', '0.500', '0.500', '0.250']

medsam_dynamic
Accuracy media: 0.450 ± 0.211
Scores per fold: ['0.200', '0.400', '0.200', '0.400', '0.400', '0.200', '0.200', '0.750', '0.750', '0.500', '0.500', '0.750', '0.750', '0.250', '0.500']

preprocessed_dynamic
Accuracy media: 0.463 ± 0.213
Scores per fold: ['0.200', '0.600', '0.200', '0.200', '0.200', '0.400', '0.400', '0.750', '0.750', '0.250', '0.500', '0.750', '0.750', '0.500', '